In [ ]:
import json # giúp python đọc và hieu được định dạng JSON trả về từ AI
import re # regular expression giúp bóc tách đúng đoạn JSON nằm giữa cặp dấu { } đề phòng AI nói dài dòng
from openai import OpenAI # Thư viện chính thức của OpenAI để gọi API
from pydantic import BaseModel # Giúp định nghĩa khuôn dữ liệu và ép kiểu dữ liệu một cách dễ dàng, tránh lỗi khi lưu vào Firebase

# 1. Định nghĩa cái khuôn dữ liệu (Giữ nguyên để kiểm tra sau khi bóc tách)
class InvoiceStructuredOutput(BaseModel): # tạo ra khuôn và quy định kiểu dữ liệu
    total_amount: float
    currency: str
 
MY_API_KEY =  "" #API Key

client = OpenAI( # kết nối với OpenRouter
    base_url="https://openrouter.ai/api/v1",
    api_key=MY_API_KEY
)

def process_ocr_with_slm(ocr_text):
    # Ép con AI bằng Prompt cực kỳ nghiêm ngặt để nó nhả đúng cấu trúc JSON
    system_instruction = (
"PRE-PROCESSING NOTE:\n"
"  - OCR text may contain mixed Traditional/Simplified Chinese due to scan errors\n"
"  - Do NOT use character script alone to determine Taiwan vs China\n"
"  - ALWAYS prioritize: phone area code > hotline format > address > script\n"
"  - 400-XXXX-XXXX hotline format = China mainland ONLY, never Taiwan\n"
"  - Area code 0371 = Zhengzhou, China = CNY\n"
"  - Taiwan area codes are: 02, 03, 04, 05, 06, 07, 08, 089 (max 2-3 digits before dash)\n"
"\n"
"CHINA vs TAIWAN disambiguation (most common confusion):\n"
"  China signals:  400-XXXX-XXXX | 0371/028/021/010-XXXXXXXX | 纳税人识别号 | 元 without NT\n"
"  Taiwan signals: 統一編號 | NT$ | 新台幣 | 02-XXXX-XXXX (8 digit local)\n"
"FALLBACK CHAIN (when no phone/address anchor exists):\n"
"  1. Tax ID format:\n"
"     - 'Mã số thuế' or 'MST' -> VND\n"
"     - 'VAT Reg' + English -> GBP or EUR (check city)\n"
"     - '統一編號' -> TWD\n"
"     - '纳税人识别号' -> CNY\n"
"     - 'GSTIN' -> INR\n"
"     - 'ABN' -> AUD\n"
"     - 'BRN' or 'SST' -> MYR\n"
"  2. Script/language of receipt text:\n"
"     - Traditional Chinese (繁體) + no CNY anchor -> TWD\n"
"     - Simplified Chinese (简体) -> CNY\n"
"     - Japanese (hiragana/katakana) -> JPY\n"
"     - Korean (hangul) -> KRW\n"
"     - Thai script -> THB\n"
"     - Arabic script -> check region (SAR/AED/EGP...)\n"
"     - Vietnamese (diacritics: ă,ơ,ư,đ) -> VND\n"
"  3. Same currency, multiple countries (USD/EUR):\n"
"     - Do NOT try to identify the exact country\n"
"     - Just confirm the ISO currency code and return it\n"
"     - EUR covers 20+ countries; USD covers USA, Ecuador, Panama, etc.\n"
"  4. Absolute last resort:\n"
"     - If zero anchors found, return: {\"total_amount\": X, \"currency\": \"UNKNOWN\"}\n"
"     - Never hallucinate a currency\n"
"\n"
"SAME CURRENCY DIFFERENT COUNTRY RULE:\n"
"  - You are extracting CURRENCY, not COUNTRY\n"
"  - If EUR is confirmed, return EUR regardless of which Eurozone country\n"
"  - If USD is confirmed, return USD regardless of which dollarized country\n"
    )
    
    # Dùng .create truyền thống (Bảo đảm không bao giờ lỗi kết nối endpoint)
    response = client.chat.completions.create( # gọi API, dịch vụ chat, kiểu xử lý completion (viết tiếp cho hoàn chỉnh vd: "total_amount": ?), hành động create là tạo một completion mới dựa trên prompt đã cho
        model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free", 
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": f"Đây là dữ liệu OCR thô: \n{ocr_text}"}
        ],
        temperature=0.0 # Ép AI phải trả về kết quả ổn định, không sáng tạo thêm gì cả (0.8-1.0 là sáng tạo, 0.0 là cực kỳ cứng nhắc)
    )
    
    # Lấy chuỗi chữ thô từ AI trả về
    raw_content = response.choices[0].message.content.strip() # response.choices[0] là lựa chọn đầu tiên (thường chỉ có 1 lựa chọn), .message.content là phần nội dung trả về, .strip() để loại bỏ khoảng trắng thừa ở đầu cuối
    
    # Dùng Regex (Bẫy bóc tách) để tìm đúng đoạn nằm giữa cặp dấu { } đề phòng AI nói dài dòng
    match = re.search(r'\{.*\}', raw_content, re.DOTALL) #tìm kiếm chuỗi bắt đầu bằng { và kết thúc bằng }, re.DOTALL để quét qua nhiều dòng nếu JSON có xuống dòng
    if match:
        json_string = match.group(0)
    else:
        json_string = raw_content

    # Ép chuỗi chữ đó thành một Object dữ liệu Pydantic
    parsed_json = json.loads(json_string)
    return InvoiceStructuredOutput(**parsed_json)

print("Đã cập nhật hàm SLM Vạn Năng thành công!")

In [ ]:
from pathlib import Path

input_path = Path('../Seg_OCR_Tri/outputs/OCR_RESULT.txt')
output_path = Path('./output/OCR_RESULT_notebook.json')

print('--- ĐANG ĐỌC OCR_RESULT.txt TỪ Seg_OCR_Tri/outputs ---')
if not input_path.exists():
    raise FileNotFoundError(f'Không tìm thấy file đầu vào: {input_path.resolve()}')

ocr_text = input_path.read_text(encoding='utf-8', errors='replace')
result = process_ocr_with_slm(ocr_text)

output_data = result.model_dump()
output_path.write_text(json.dumps(output_data, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Đã xuất JSON vào: {output_path.resolve()}')
print(json.dumps(output_data, ensure_ascii=False, indent=2))

In [ ]:
# Giả lập một cục text cực kỳ lộn xộn, lỗi font, sai chính tả do tầng OCR quét từ ảnh sang
mock_ocr_output = """
📝 KẾT QUẢ SAU KHI GOM DÒNG HOÀN CHỈNH:
============================================================
HÓA ĐƠN THANH Số TOÁN HD: SVHOO1 5 Ma HĐ: #JKKBE Bàn; MANG VE TN; Ca Sang 830 SVH Giò vào: 12.24 Ngày; 08/06/2026 GIò ra: 1225 StT Tên món SL Đon Thanh Trà glá Olong tíen Đol Xoài Nhiệt (L) 39,000 39,000 Ít Đường OFF (CT PHỤ) - Mua 01 ly size L lặng châu Trân Òlong Trân Châu Ôlong Thành tiền: 39,000 @ Tổng tiêr: 39,000 đ +Thanh toán tiền mặt 39,000 4 Tiên nhận 50,000 9 Tíền thừa 11,000 4 OLONG CHMU 830 Sư Vạn Hạnh, 910 Đja chỉ: 830 Sư Vận Hạnh, Phường 13, Quận 10, Hồ Chí Minh
GlỮ HÓA ĐƠN NHẬN UU ĐÃl
IĂNG 01 TOPPING KHIMUA SẢN_PHẢM BẨIKỲ
Ưu đấi áp dung trong 2 ngày
(tính từ ngày xuất hóa đơn)
Khòng áp dung cùng lúc các ưu đãi khác
Số lượng ưu đãi có han mẫi hóa dơn chi tặng 01 Phần quà
Cảm ơn Quý Khách Powered by iPOS. vn
============================================================
"""

print("--- Đang gửi dữ liệu thô sang cho bộ não SLM phân tích... ---")

# Gọi hàm xử lý
result = process_ocr_with_slm(mock_ocr_output)

print("\n--- KẾT QUẢ ĐÃ ĐƯỢC SLM ÉP KHUÔN JSON THÀNH CÔNG: ---")
# Dùng model_dump() thay cho dict() để không bị báo lỗi cảnh báo màu vàng nữa nhé bạn
print(json.dumps(result.model_dump(), ensure_ascii=False, indent=4))

print("\n--- KIỂM TRA KIỂU DỮ LIỆU ĐỂ LƯU VÀO FIREBASE: ---")
print(f"Số tiền trích xuất: {result.total_amount} (Kiểu dữ liệu: {type(result.total_amount)})")
print(f"Đơn vị tiền tệ: {result.currency} (Kiểu dữ liệu: {type(result.currency)})")

In [ ]:
# Giả lập một cục text cực kỳ lộn xộn, lỗi font, sai chính tả do tầng OCR quét từ ảnh sang
mock_ocr_output = """
--- KẾT QUẢ OCR HÓA ĐƠN HOÀN CHỈNH ---
📝 Hộ KINH DOANH CƠM GA NHA TRANG 2 CHJ EM ĐC 09 Thich MInh Nguyet; Phường Tân Sơn Hòa, TPHCM MST: 082089009531 HOTLINE: 0906619578
📝 Bàn 8
📝 PHIẾU TÍNH TIỀN HD035530
📝 'Giờ vào 07/06/2026 19.57
📝 Giờ in: 20,17 Thành Tên hàng Đ.Giá SL tien chương trinh khuyến mãi Coca Od 0 x 3 Cơm gà ta xé lớn (Phần) 58,000 x 2 116,000 Cơm xá xíu (Phần) 59,000 x 59,000 Sất trứng thêm (Phần) 7,000 7,000 Canh soup trứng tlem (Phần) 5,000 X 5,000 Tông ti`n hàng: 187,000 Chiết khấu: 187,000 TWng Cộng: Chuyen khoản Cảm ơn quý khách và hen gặp lai III Powered by KiotViet
"""

print("--- Đang gửi dữ liệu thô sang cho bộ não SLM phân tích... ---")

# Gọi hàm xử lý
result = process_ocr_with_slm(mock_ocr_output)

print("\n--- KẾT QUẢ ĐÃ ĐƯỢC SLM ÉP KHUÔN JSON THÀNH CÔNG: ---")
# Dùng model_dump() thay cho dict() để không bị báo lỗi cảnh báo màu vàng nữa nhé bạn
print(json.dumps(result.model_dump(), ensure_ascii=False, indent=4))

print("\n--- KIỂM TRA KIỂU DỮ LIỆU ĐỂ LƯU VÀO FIREBASE: ---")
print(f"Số tiền trích xuất: {result.total_amount} (Kiểu dữ liệu: {type(result.total_amount)})")
print(f"Đơn vị tiền tệ: {result.currency} (Kiểu dữ liệu: {type(result.currency)})")

In [ ]:
# Giả lập một cục text cực kỳ lộn xộn, lỗi font, sai chính tả do tầng OCR quét từ ảnh sang
mock_ocr_output = """
--- KẾT QUẢ OCR HÓA ĐƠN HOÀN CHỈNH ---
📝 一好彩揩肚鴉火鍋{錄地30店} 
📝 大廳
📝 預結單1 
📝 台號: 01 01 賬單號: 10016211002000052 收銀員 : 3002 收銀員 值台員: -999 線上支付 開旨時間 2021-10-02 19:31:09 預結時間 # 2021~10-02 21:37:23
📝 數:2
📝 菜品名稱
📝 數量
📝 單併
📝 金額
📝 1.小料 2 &.00 `2.00 2.招牌褚肚鴉(小] ;8.00 88.00 3.野筍片 .00 16.00 4.蟹籽蝦滑{小)  .00 28 .00 5.鮮牛肉 ~.36.00 36.00 6.竹笙 _32. 00 32.00 7.腊味煲仔皈 35.00 35.00 消費台計 247.00 應付全頡 247.00 好彩渚肚鵝火鍋{綠地360店]祝您用餐恂快! 由話: 0371-53320888 地址: 綠地新都  330廣場 好彩堵肚鴉火鍋于2018年11月8日與大家[面, 至兮遍 布中原32地,一既成為2020年大灰點評必吃榜品牌! 經過2年的市場洗禮,于業卑率先提出%栲統]`式k鍋 喝湯暖胃訃身廿"的火鍋新主張. 堅持文火慢熬8小 時,堅持采用海南白胡椒,堅持廣東人吃鵡塊蘸姜蓉的 標配吃法, 堅持堵肚蘸海鮮汁的傳統吃法,由此抓起一 場堵肚雞火鍋小料的吃法六f. 更多精彩可夫注奸彩堵肚鴉火鍋公眾號. [好彩唯一熱線400-0701-4001
"""

print("--- Đang gửi dữ liệu thô sang cho bộ não SLM phân tích... ---")

# Gọi hàm xử lý
result = process_ocr_with_slm(mock_ocr_output)

print("\n--- KẾT QUẢ ĐÃ ĐƯỢC SLM ÉP KHUÔN JSON THÀNH CÔNG: ---")
# Dùng model_dump() thay cho dict() để không bị báo lỗi cảnh báo màu vàng nữa nhé bạn
print(json.dumps(result.model_dump(), ensure_ascii=False, indent=4))

print("\n--- KIỂM TRA KIỂU DỮ LIỆU ĐỂ LƯU VÀO FIREBASE: ---")
print(f"Số tiền trích xuất: {result.total_amount} (Kiểu dữ liệu: {type(result.total_amount)})")
print(f"Đơn vị tiền tệ: {result.currency} (Kiểu dữ liệu: {type(result.currency)})")

In [ ]:
# Kiểm tra model/endpoints khả dụng với client hiện tại
print('--- Kiểm tra model/endpoints khả dụng ---')
try:
    models = client.models.list()
    print('Found models:')
    for m in getattr(models, 'data', models):
        # Một số SDK trả về dict-like, một số trả về object có .data
        mid = getattr(m, 'id', m.get('id') if isinstance(m, dict) else str(m))
        print('-', mid)
except Exception as e:
    print('Không thể liệt kê model:', repr(e))

In [ ]:
# Giả lập một cục text cực kỳ lộn xộn, lỗi font, sai chính tả do tầng OCR quét từ ảnh sang
mock_ocr_output = """
HÓA ĐƠN THANH TOÁN Sđ HD: SVHOO15 Má HĐ; #JKKBE TN: Ca Sáng 830 SVH Bàn; MANG vỀ Ngày; 08/06/2026 Glò vào: : 12.24 Glò ra: 12.25 Stt Tên món SL Đơn Thanh glá tlòn Trà ỗlong Xoài Nhiệt Đới (L) 1 39,000 39,000 Ít Đuờng OFF (CT PHU) Mua 2 01 ly size L lặng Trân 1 châu Olong Trân Châu Ôlong Thành tên: 39,000@ Tổng tiêr: 39,000 4 +Thanh toán tiền mặt 39,000 4 Tiên nhận 50,000 4 Tỉền thừa 11,000 4 OLONG CHAU 830 Sư Van Hanh, Q10 Đja chỈ: 830 Sư Vạn Hạnh Phường 13, Quặ Hồ Chí Minh
GlỮ HÓA ĐƠN NHẬN UU ĐẴl
IXNG@1 IOPPING KHLMUASANPHẢMBÄI Ưu đãi áp dụng trong 2 ngày (tính từ ngày xuất hóa đơn) Khòng áp dyng cung lúc các ưu đãí khấc Sõ Iượng ưu đãi có han mẫi hóa dơn chỉ tặng 01 phần quà
Cám ơn Quý Khách Powered by iPOS.vn
"""

print("--- Đang gửi dữ liệu thô sang cho bộ não SLM phân tích... ---")

# Gọi hàm xử lý
result = process_ocr_with_slm(mock_ocr_output)

print("\n--- KẾT QUẢ ĐÃ ĐƯỢC SLM ÉP KHUÔN JSON THÀNH CÔNG: ---")
# Dùng model_dump() thay cho dict() để không bị báo lỗi cảnh báo màu vàng nữa nhé bạn
print(json.dumps(result.model_dump(), ensure_ascii=False, indent=4))

print("\n--- KIỂM TRA KIỂU DỮ LIỆU ĐỂ LƯU VÀO FIREBASE: ---")
print(f"Số tiền trích xuất: {result.total_amount} (Kiểu dữ liệu: {type(result.total_amount)})")
print(f"Đơn vị tiền tệ: {result.currency} (Kiểu dữ liệu: {type(result.currency)})")